In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR

from xgboost import XGBRegressor


n_repeats = 50
test_size = 0.2

noise_multiplier = 10
noise_level_feat = 0.1
noise_level_target = 0.05

xgb_params = dict(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

svr_params = dict(
    kernel="rbf",
    C=100.0,
    epsilon=0.1,
    gamma="scale",
)


def build_preprocessor(X_fit_like, for_svr):
    numeric_cols = X_fit_like.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_cols = X_fit_like.select_dtypes(include=["object"]).columns.tolist()

    if for_svr:
        num_tf = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
    else:
        num_tf = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ])

    cat_tf = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_tf, numeric_cols),
            ("cat", cat_tf, categorical_cols),
        ]
    )


def make_noise_copy(Xy, num_cols, target_col, seed):
    rng = np.random.default_rng(seed)
    Xy_noise = Xy.copy()

    for col in num_cols:
        std = Xy[col].std()
        noise = rng.normal(loc=0.0, scale=noise_level_feat * std, size=len(Xy))
        Xy_noise[col] = Xy[col] + noise

    std_y = Xy[target_col].std()
    noise_y = rng.normal(loc=0.0, scale=noise_level_target * std_y, size=len(Xy))
    Xy_noise[target_col] = Xy[target_col] + noise_y

    return Xy_noise


def make_noise_trainset(X_train, y_train, seed):
    Xy = X_train.copy()
    Xy[target_col] = y_train

    num_cols = Xy.select_dtypes(include=["int64", "float64"]).columns.tolist()
    num_cols = [c for c in num_cols if c != target_col]

    copies = [Xy]
    n_new_copies = noise_multiplier - 1

    for i in range(n_new_copies):
        copies.append(make_noise_copy(Xy, num_cols, target_col, seed + i + 1))

    Xy_full = pd.concat(copies, axis=0).reset_index(drop=True)

    X_out = Xy_full.drop(columns=[target_col])
    y_out = Xy_full[target_col]

    return X_out, y_out


def eval_model(X_train_like, y_train_like, X_test, y_test, model_name):
    if model_name == "xgb":
        pre = build_preprocessor(X_train_like, for_svr=False)
        model = XGBRegressor(**xgb_params)
    else:
        pre = build_preprocessor(X_train_like, for_svr=True)
        model = SVR(**svr_params)

    pipe = Pipeline(steps=[
        ("preprocessor", pre),
        ("model", model),
    ])

    pipe.fit(X_train_like, y_train_like)
    pred = pipe.predict(X_test)

    mse = mean_squared_error(y_test, pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    return float(mse), rmse, float(mae), float(r2)


rows = []
split_seeds = list(range(1000, 1000 + n_repeats))

for rep, rs in enumerate(split_seeds, 1):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=rs,
        shuffle=True,
    )

    X_train_noise, y_train_noise = make_noise_trainset(X_train, y_train, seed=rs + 22)

    xgb_mse, xgb_rmse, xgb_mae, xgb_r2 = eval_model(X_train_noise, y_train_noise, X_test, y_test, "xgb")
    svr_mse, svr_rmse, svr_mae, svr_r2 = eval_model(X_train_noise, y_train_noise, X_test, y_test, "svr")

    rows.append({
        "repeat": rep,
        "split_seed": rs,
        "xgb_mse": xgb_mse,
        "xgb_rmse": xgb_rmse,
        "xgb_mae": xgb_mae,
        "xgb_r2": xgb_r2,
        "svr_mse": svr_mse,
        "svr_rmse": svr_rmse,
        "svr_mae": svr_mae,
        "svr_r2": svr_r2,
    })

results_noise10x = pd.DataFrame(rows)


def summarize(series):
    q1 = float(series.quantile(0.25))
    q3 = float(series.quantile(0.75))
    desc = series.describe()

    return pd.Series({
        "count": int(desc["count"]),
        "mean": float(desc["mean"]),
        "std": float(desc["std"]),
        "min": float(desc["min"]),
        "q1": q1,
        "median": float(series.median()),
        "q3": q3,
        "iqr": q3 - q1,
        "max": float(desc["max"]),
    })


summary_noise10x = pd.DataFrame({
    "xgb_r2": summarize(results_noise10x["xgb_r2"]),
    "svr_r2": summarize(results_noise10x["svr_r2"]),
}).T

display(results_noise10x.head())
display(summary_noise10x)

results_noise10x.to_csv("noise10x_repeated_split_results.csv", index=False)
summary_noise10x.to_csv("noise10x_repeated_split_summary.csv", index=True)